# Unsupervised Machine Learning Project

Ka Ho CHAN 9930788

## Set Up

In [25]:
import pandas as pd                                         # tables (dataframes)
import numpy as np
import time
import tracemalloc                                          # measure peak memory (also tracks numpy arrays)
from mlxtend.preprocessing import TransactionEncoder        # one hot encoding: baskets > T/F table
from mlxtend.frequent_patterns import apriori               # find the frequent itemsets
from mlxtend.frequent_patterns import association_rules     # find the association rules

'''
Run build() and return (result, seconds, peak memory used in MB).
- seconds : wall-clock time of the run
- peak MB : the highest memory reached during the run (not just what is left at the end)
- track_memory=False : time only, no tracing (peak MB is returned as None)

Why two modes: tracemalloc records every memory allocation. Code that makes millions of small
Python allocations (the naive groupby loop) runs several times slower while traced, which would
make its time look worse than it really is. Code that makes a few large numpy allocations is barely
affected. So for a fair comparison, time is taken from an untraced run and peak memory from a traced run.

Used in Phase 1 for measuring the performance of different approaches.
Used in Phase 2 to measure every Apriori run (numpy-heavy, so one traced run is enough there).
'''
def measure(build, track_memory=True):
    if track_memory:
        tracemalloc.start()
    start = time.perf_counter()
    result = build()
    seconds = time.perf_counter() - start
    peak_mb = None
    if track_memory:
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        peak_mb = peak / 1e6
    return result, seconds, peak_mb

## Phase 1 — Load and prepare (the wide format) 

### Phase 1.1 - Load and Verify the transaction and product files

In [26]:
'''
The file has ONE ROW PER LINE ITEM (not one row per basket).
Several rows share the same TransactionID: those rows together are one receipt. 
'''

df = pd.read_csv("../data/transactions.zip", usecols=["TransactionID", "Date", "Item"])
products = pd.read_csv("../data/products.csv")

print('File Loaded.')
print(f'     rows (line items):   {len(df):,}')
n_baskets = df['TransactionID'].nunique()
print(f'     transactions:        {n_baskets:,}')
print(f'     columns:             {list(df.columns)}') # list gives better formatting
print(f'     products:            {df["Item"].nunique()}')
print(f'     average basket size: {len(df) / n_baskets:.2f}')

# Show that the data is clean and consistent
print('\nShow that the data is clean and consistent:')
print(f'     missing values:      {df.isnull().sum().sum()}')
print(f'     date range:          {df["Date"].min()} -> {df["Date"].max()}')
print(f'     same item twice in a basket: {df.duplicated(["TransactionID", "Item"]).sum()}')
print(f'     names not in products.csv:   {len( set(df['Item']) - set(products['Item']))} (0 = all names match the catalogue)')

print('\nShow the first 5 rows of the data:')
print(df.head())

File Loaded.
     rows (line items):   8,379,837
     transactions:        1,000,000
     columns:             ['TransactionID', 'Date', 'Item']
     products:            98
     average basket size: 8.38

Show that the data is clean and consistent:
     missing values:      0
     date range:          2026-01-01 -> 2026-12-31
     same item twice in a basket: 0
     names not in products.csv:   0 (0 = all names match the catalogue)

Show the first 5 rows of the data:
   TransactionID        Date                 Item
0         100000  2026-06-19       Cidre de glace
1         100000  2026-06-19              Couches
2         100000  2026-06-19      Fromage cheddar
3         100000  2026-06-19  Lingettes pour bébé
4         100000  2026-06-19      Purée pour bébé


### Phase 1.2a - Organize the data in wide format: one list of items per transaction

In [27]:
'''
Naive method (as we used in the exercise): group the rows by TransactionID, then TransactionEncoder.
- groupby + lambda builds one Python list per basket -> 1,000,000 lists of strings in memory
- TransactionEncoder then converts those lists into the True/False table
This is a Python loop over 1M groups, so it will be slow, 
and the lists use far more memory than the final table.
'''
def build_naive(df):
    # return a list of sorted unique items for each transaction
    baskets = df.groupby('TransactionID')['Item'].apply(lambda items: sorted(set(items))).tolist()
    te = TransactionEncoder()
    return pd.DataFrame(te.fit(baskets).transform(baskets), columns=te.columns_)

# run 1: time only (no tracing, so the Python loop is not slowed down)
_, time_naive, _ = measure(lambda: build_naive(df), track_memory=False)
# run 2: peak memory (traced; its time is not used because tracing slows the loop)
basket_table_naive, time_naive_traced, peak_naive = measure(lambda: build_naive(df))

print('Phase 1.2a - naive method (groupby + TransactionEncoder)')
print(f'    time        : {time_naive:.1f} s   (untraced run; {time_naive_traced:.1f} s with tracemalloc on)')
print(f'    peak memory : {peak_naive:,.0f} MB')
print(f'    table       : {basket_table_naive.shape}, {basket_table_naive.memory_usage(deep=True).sum() / 1e6:.0f} MB')

Phase 1.2a - naive method (groupby + TransactionEncoder)
    time        : 21.5 s   (untraced run; 90.8 s with tracemalloc on)
    peak memory : 920 MB
    table       : (1000000, 98), 98 MB


### Phase 1.2b - Fast Approach: numpy function without a Python loop

In [28]:
'''
Fast method: no Python loop.
- pd.factorize turns every TransactionID into a row number (0..999,999)
  and every Item into a column number (0..97)
- one numpy assignment then sets all 8.4M True values at once
- dtype=bool -> 1 byte per cell (int64 would be 8 bytes per cell = 784 MB)
'''
def build_fast(df):
    basket_codes, basket_ids = pd.factorize(df['TransactionID'])
    item_codes, item_names = pd.factorize(df['Item'])
    matrix = np.zeros((len(basket_ids), len(item_names)), dtype=bool)
    matrix[basket_codes, item_codes] = True
    return pd.DataFrame(matrix, index=basket_ids, columns=item_names).sort_index(axis=1)

# same two-run protocol as the naive method, so the comparison is fair
_, time_fast, _ = measure(lambda: build_fast(df), track_memory=False)
basket_table, time_fast_traced, peak_fast = measure(lambda: build_fast(df))

print('Phase 1.2b - fast method (factorize + numpy)')
print(f'    time        : {time_fast:.2f} s   (untraced run; {time_fast_traced:.2f} s with tracemalloc on)')
print(f'    peak memory : {peak_fast:,.0f} MB')
print(f'    table       : {basket_table.shape}, {basket_table.memory_usage(deep=True).sum() / 1e6:.0f} MB '
      f'(1 byte x 98M cells + 8 MB basket index)')
print(f'    same table as naive : {(basket_table.sort_index().to_numpy() == basket_table_naive.to_numpy()).all()}')
print(f'    speed-up    : {time_naive / time_fast:.0f}x faster, peak memory {peak_naive / peak_fast:.1f}x lower')

del basket_table_naive   # free memory before Apriori

Phase 1.2b - fast method (factorize + numpy)
    time        : 0.71 s   (untraced run; 0.70 s with tracemalloc on)
    peak memory : 436 MB
    table       : (1000000, 98), 106 MB (1 byte x 98M cells + 8 MB basket index)
    same table as naive : True
    speed-up    : 30x faster, peak memory 2.1x lower


#### First Genuine Engineering Problem

| Method | Time | Peak memory | Final table |
|---|---|---|---|
| groupby + TransactionEncoder (naive method) | 21.5 s | 920 MB | 98 MB |
| factorize + numpy (fast method) | 0.71 s | 436 MB | 106 MB |

- **Problem:** Apriori needs a boolean table with one row per basket and one column per product (1,000,000 * 98). The number of rows (line items) in the file is 8,379,837
- **Why the naive method is costly:** it builds 1,000,000 Python lists of strings before encoding.
  Its peak memory is 9.4x the size of the final table, and the Python loop over 1M groups is 30x slower than the fast method. 
- **Why the fast method works:** `pd.factorize` maps baskets and products to row/column numbers,
  and a single numpy assignment fills the table with no Python loop.
- **Memory depends mostly on the dtype:** as `bool` the table takes 98 MB; the same table in `int64`
  (e.g. from `pd.get_dummies` or `pd.crosstab` counts) would take 784 MB.
- **Decision:** both methods fit on this machine (32 GB RAM), and both give the identical table.
  The fast method is kept because it is about 30 times faster with a lower peak of 2.1 times, and Phase 4 rebuilds the table for two more seasons.

### Phase 1.3 - Find the item frequencies

In [29]:
item_freq = basket_table.mean().sort_values(ascending=False)   # support of each single product
freq_table = (item_freq.rename('support').to_frame()
              .assign(n_baskets=(item_freq * len(basket_table)).round().astype(int))
              .join(products.set_index('Item')[['Item_EN', 'Category']]))

print('10 most frequent products'); display(freq_table.head(10))
print('5 rarest products');        display(freq_table.tail(5))

10 most frequent products


,support,n_baskets,Item_EN,Category
Sac réutilisable,0.900026,900026,Reusable bag,Maison
Biscuits,0.161439,161439,Cookies,Collations
Noix mélangées,0.155189,155189,Mixed nuts,Collations
Beurre d'arachide,0.153726,153726,Peanut butter,Épicerie
Bière,0.150468,150468,Beer,Boissons
Savon à vaisselle,0.149820,149820,Dish soap,Maison
Fromage cheddar,0.143457,143457,Cheddar cheese,Fromages
Charbon de bois,0.140561,140561,Charcoal,Épicerie
Papier hygiénique,0.139758,139758,Toilet paper,Maison
Céréales,0.139198,139198,Cereal,Épicerie


5 rarest products


,support,n_baskets,Item_EN,Category
Safran,0.000818,818,Saffron,Spécialités
Huile de truffe,0.000817,817,Truffle oil,Spécialités
Caviar,0.000810,810,Caviar,Spécialités
Vinaigre balsamique vieilli,0.000779,779,Aged balsamic vinegar,Spécialités
Foie gras,0.000756,756,Foie gras,Spécialités


#### Observation in Phase 1.3

- The support of Sac réutilisable is very high at 90%, while the support of the next product is at 16%. 
- The support of the 5 rarest products are around 0.08% which is below every min_support tested (5%, 2%, 1% and 0.5%). Apriori will therefore find no rule containing them, so the app can never recommend them.

### Phase 1.4 - Prediction of which product will cause trouble
(Written before running Apriori, from the frequencies in Phase 1.3)
- **Product:** Sac réutilisable
- **Reason:** It is in 90% of the baskets, so almost any rule saying "X → Sac réutilisable" will have a confidence near 0.90. But lift = confidence / support(Sac réutilisable) ≈ 0.90 / 0.90 ≈ 1, meaning that the rule tells nothing to us. 
- **Expected result:** It will fill the top 10 list by confidence and be absent from the top 10 list by lift. To be checked in Phase 3. 
- **Result:** See the result in Phase 3.3. The Phase 1 prediction for the top confidence rules suggesting Sac réutilisable is correct. And the prediction for Sac réutilisable absent from the top 10 rules by lift is correct.  

### Phase 1.5 - Evidence of a duplicated product

In [30]:
a, b = 'Baguette', 'Pain baguette rustique'
display(products[products['Item'].isin([a, b])])
both = (basket_table[a] & basket_table[b]).sum()
expected = basket_table[a].mean() * basket_table[b].mean() * len(basket_table)
print(f'bought together: {both:,}   expected if unrelated: {expected:,.0f}')

pair = df[df['Item'].isin([a, b])]
display(pair.groupby([pair['Date'].str[5:7], 'Item']).size().unstack())   # sales by month

,Item,Item_EN,Category,Price
48,Baguette,Baguette,Boulangerie,2.99
96,Pain baguette rustique,Rustic baguette,Boulangerie,3.99


bought together: 3,009   expected if unrelated: 2,930


Item,Baguette,Pain baguette rustique
Date,,
01,5985,5691
02,5342,4813
03,5845,3851
04,5697,2614
05,5894,2164
06,5584,1757
07,5943,1798
08,6047,1905
09,5706,2366


#### **Decision to handle these products**
- The sale of baguette is almost the same throughout the year. 
- However, the sale of rustic baguette is higher during the months with lower temperature and lower during the months with higher temperature. 
- These two are essentially different products. Therefore, they will be kept as separate products. 

## Phase 2 — Mine the rules with Apriori

Steps: first run `apriori()` with its default settings, as given in the brief, from the highest to the lowest min_support, to meet the MemoryError. Then fix it with `low_memory=True` and run all four min_support values (0.05, 0.02, 0.01, 0.005), recording the itemsets, rules, time and peak memory of each run.

### Phase 2.1 - Default settings: meeting the MemoryError

In [ ]:
'''
apriori() with its DEFAULT settings (low_memory=False), one cell per min_support,
from the highest to the lowest, to see at which value the memory runs out.
The notebook is saved before running each cell: a killed process loses everything not written to disk.
'''
# min_support = 0.05 with the default settings
min_support = 0.05
try:
    itemsets, seconds, peak_mb = measure(lambda: apriori(basket_table, min_support=min_support, use_colnames=True, low_memory=False))
    print(f'min_support={min_support}: OK, {len(itemsets)} itemsets, {seconds:.1f} s, peak memory {peak_mb:,.0f} MB')
except MemoryError as error:
    tracemalloc.stop()   # the error stopped measure() before its own tracemalloc.stop()
    print(f'min_support={min_support}: MemoryError -> {error}')

min_support=0.05: OK, 191 itemsets, 4.5 s, peak memory 7,676 MB


In [ ]:
# min_support = 0.02 with the default settings
min_support = 0.02
try:
    itemsets, seconds, peak_mb = measure(lambda: apriori(basket_table, min_support=min_support, use_colnames=True, low_memory=False))
    print(f'min_support={min_support}: OK, {len(itemsets)} itemsets, {seconds:.1f} s, peak memory {peak_mb:,.0f} MB')
except MemoryError as error:
    tracemalloc.stop()   # the error stopped measure() before its own tracemalloc.stop()
    print(f'min_support={min_support}: MemoryError -> {error}')

min_support=0.02: OK, 658 itemsets, 10.2 s, peak memory 12,842 MB


In [ ]:
# min_support = 0.01 with the default settings
min_support = 0.01
try:
    itemsets, seconds, peak_mb = measure(lambda: apriori(basket_table, min_support=min_support, use_colnames=True, low_memory=False))
    print(f'min_support={min_support}: OK, {len(itemsets)} itemsets, {seconds:.1f} s, peak memory {peak_mb:,.0f} MB')
except MemoryError as error:
    tracemalloc.stop()   # the error stopped measure() before its own tracemalloc.stop()
    print(f'min_support={min_support}: MemoryError -> {error}')

min_support=0.01: OK, 2267 itemsets, 43.5 s, peak memory 24,682 MB


In [ ]:
# min_support = 0.005 with the default settings
min_support = 0.005
try:
    itemsets, seconds, peak_mb = measure(lambda: apriori(basket_table, min_support=min_support, use_colnames=True, low_memory=False))
    print(f'min_support={min_support}: OK, {len(itemsets)} itemsets, {seconds:.1f} s, peak memory {peak_mb:,.0f} MB')
except MemoryError as error:
    tracemalloc.stop()   # the error stopped measure() before its own tracemalloc.stop()
    print(f'min_support={min_support}: MemoryError -> {error}')

min_support=0.005: OK, 7106 itemsets, 562.4 s, peak memory 108,187 MB


#### Why the default settings fail, and the fix

- I have executed twice with min_support = 0.005 using the cell above. At the first time, the process raised the memory error when the process needed to allocate 78 GB of RAM for the shape of the data array on a local PC which has only 32 GB of RAM. Then, when I executed for the second time, the PC was close to be frozen while it ran. But after almost 10 mins, the cell stopped and showed the total time spent and the peak memory request of 108 GB. 
- I may have fallen into a trap of expecting the memory error at each time the same cell was run. But I kept the honest result of intermittent memory errors when may not occur all the time. 
- My guess for the success at the second time is that the Windows OS has managed to use physical memory (RAM) and allocate enough virtual memory as well on the hard drive. So the peak memory appeared to be larger than just the physical memory size. Also, I guess that Windows was able to compress pages of the Apriori's arrays which are mostly false in the True/False table. 
- The reason for the peak memory exploding is that the default mode (low_memory = False) checks all candidates at once in one array of shape (candidates, k, baskets); if the shape is (candidates=25975, k=3, baskets=1000000), it means 25,975 triples × 3 × 1,000,000 = about 78 GB. Therefore, lower min_support means more candidates, so memory climbs and time climbs with it too. 
- The setting of low_memory = True should fix it because it will minimize the amount of memory usage and I will test it in Phase 2.2

### Phase 2.2 - The fix: low_memory=True at four min_support values

In [31]:
'''
Same apriori() call with low_memory=True: the candidate itemsets are checked one at a time
instead of all at once in one huge array, so the memory stays small.

association_rules(): keep the rules with a confidence of at least 0.2
(the THEN product is in at least 20% of the baskets that contain the IF products).
'''
runs = {}      # min_support -> (itemsets, rules), reused in Phase 3
results = []   # one row per run, for the table in reports/benchmark.md

for min_support in [0.05, 0.02, 0.01, 0.005]:
    itemsets, seconds, peak_mb = measure(
        lambda: apriori(basket_table, min_support=min_support, use_colnames=True, low_memory=True))
    rules = association_rules(itemsets, metric='confidence', min_threshold=0.2)
    runs[min_support] = (itemsets, rules)
    results.append({'min_support': min_support,
                    'itemsets': len(itemsets),
                    'rules': len(rules),
                    'time (s)': round(seconds, 1),
                    'peak memory (MB)': round(peak_mb)})

benchmark = pd.DataFrame(results)
benchmark

,min_support,itemsets,rules,time (s),peak memory (MB)
0,0.050,191,282,2.0,112
1,0.020,658,3159,4.8,115
2,0.010,2267,9736,11.3,115
3,0.005,7106,28433,26.2,115


#### Observations on the four runs

- **Making the min_support lower will make the output explode:** When the min_support is lowered to each of the values above, the itemsets size grow by about 3 times. Going from 0.05 to 0.005 in min_support, the itemsets size will be multiplied by 37. Rules grow even faster at 100 times. 
- **Time grows with the number of itemsets:** The time goes from 2 to 26.2 seconds, about 13 times more because of the larger itemsets. 
- **Memory stays almost the same:** The key result is the peak memory size stays almost the same. With low_memory = True, the memory does not depend on the min_support (memory = baskets × products per candidate) because only one candidate is checked at a time; with the default mode, the memory usage is much larger with (memory = candidates × baskets × products per candidate).  

## Phase 3 — Evaluate the rules

### Phase 3.1 - Definitions
- **Support:** measures how often an item appears across all baskets.
- **Confidence:** measures how often B appears in a basket that already contains A.
- **Lift:** compares the confidence with B's normal popularity, i.e. buying A makes B more likely to appear in the basket than usual or not.  

### Phase 3.2 - Top 10 simple rules by confidence and by lift

In [32]:
'''
The rules of the lowest min_support (0.005) are used from here on: the most rules, so the widest view.
'''
final_support = 0.005
itemsets, all_rules = runs[final_support]

def simple_rules(rules):
    # keep only the rules with ONE product on each side (IF one product -> THEN one product)
    return rules[(rules['antecedents'].apply(len) == 1) & (rules['consequents'].apply(len) == 1)]

def readable(rules):
    # frozenset({'A', 'B'}) -> 'A + B', and keep only the columns needed
    table = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
    table['antecedents'] = table['antecedents'].apply(lambda items: ' + '.join(sorted(items)))
    table['consequents'] = table['consequents'].apply(lambda items: ' + '.join(sorted(items)))
    return table.round(3).reset_index(drop=True)

simple = simple_rules(all_rules)
print(f'{len(simple)} simple rules out of {len(all_rules):,} rules')

top_confidence = readable(simple.sort_values('confidence', ascending=False).head(10))
top_lift = readable(simple.sort_values('lift', ascending=False).head(10))

print('\nTop 10 simple rules by CONFIDENCE')
display(top_confidence)
print('Top 10 simple rules by LIFT')
display(top_lift)

399 simple rules out of 28,433 rules

Top 10 simple rules by CONFIDENCE


,antecedents,consequents,support,confidence,lift
0,Concombre,Sac réutilisable,0.063,0.903,1.003
1,Cidre de glace,Sac réutilisable,0.053,0.902,1.002
2,Sauce tomate,Sac réutilisable,0.096,0.902,1.002
3,Relish,Sac réutilisable,0.053,0.902,1.002
4,Huile d'olive,Sac réutilisable,0.049,0.902,1.002
5,Pommes,Sac réutilisable,0.082,0.902,1.002
6,Moutarde de Dijon,Sac réutilisable,0.074,0.902,1.002
7,Fromage suisse,Sac réutilisable,0.084,0.901,1.002
8,Steak de boeuf,Sac réutilisable,0.095,0.901,1.002
9,Avocat,Sac réutilisable,0.040,0.901,1.001


Top 10 simple rules by LIFT


,antecedents,consequents,support,confidence,lift
0,Fromage à raclette,Bouillon à fondue,0.034,0.491,10.171
1,Bouillon à fondue,Fromage à raclette,0.034,0.697,10.171
2,Charcuteries,Fromage à raclette,0.032,0.683,9.976
3,Fromage à raclette,Charcuteries,0.032,0.466,9.976
4,Fromage à raclette,Cornichons,0.030,0.434,9.773
5,Cornichons,Fromage à raclette,0.030,0.670,9.773
6,Fromage à raclette,Pain baguette rustique,0.028,0.403,9.552
7,Pain baguette rustique,Fromage à raclette,0.028,0.654,9.552
8,Algues nori,Riz à sushi,0.054,0.795,9.505
9,Riz à sushi,Algues nori,0.054,0.651,9.505


### Phase 3.3 - Checking the Phase 1 prediction

In [33]:
# How many rules in each top 10 have the reusable bag on the THEN side?
bag = 'Sac réutilisable'
print(f'top 10 by confidence: {(top_confidence["consequents"] == bag).sum()} of 10 rules suggest {bag}')
print(f'top 10 by lift:       {(top_lift["consequents"] == bag).sum()} of 10 rules suggest {bag}')

# All the simple rules "X -> Sac réutilisable": their confidence and lift
bag_rules = readable(simple[simple['consequents'] == frozenset([bag])])
print(f'\n{len(bag_rules)} simple rules suggest {bag}:')
print(f'    confidence from {bag_rules["confidence"].min()} to {bag_rules["confidence"].max()}')
print(f'    lift       from {bag_rules["lift"].min()} to {bag_rules["lift"].max()}')

top 10 by confidence: 10 of 10 rules suggest Sac réutilisable
top 10 by lift:       0 of 10 rules suggest Sac réutilisable

92 simple rules suggest Sac réutilisable:
    confidence from 0.897 to 0.903
    lift       from 0.997 to 1.003


### Phase 3.4 - Comparison of the two rankings

- The Phase 1 prediction for the 10 of 10 top confidence rules suggesting Sac réutilisable is correct. And the prediction for Sac réutilisable absent from the top 10 rules by lift is correct.  
- All top 10 rules by confidence suggest Sac réutilisable. The IF side is different items, but the THEN side is always Sac réutilisable. Confidence ranges from 0.901 to 0.903. This is because Sac réutilisable is in 90% of the baskets, as found in Phase 1.3. Confidence measures how often B (Sac réutilisable) appears in a basket that already contains A (any other items). When Sac réutilisable is in 90% of the baskets, it is also in 90% of the baskets with any other items. However, the lift of those rules is all around 1.00 which means that those rules tell us nothing. The Sac réutilisable is not bought because of A but because it is just bought by almost everyone. 
- The top 10 rules by lift suggests Fromage à raclette, Bouillon à fondue, Charcuteries, Cornichons and Pain baguette rustique which are all the ingredients of a meal. Lift of 9.5 to 10.2 suggest that those ingredients are bought together about 10 times more often than by chance. The confidence is lower between 0.40 to 0.70. Each rule appears in both direction (A → B and B → A) with the same value of lift but different confidence. This shows me that lift is symmetric but confidence is not. None of those rules by lift suggest the bag. 
- Suggesting the bag by the confidence ranking adds no value, because 90% of customers buy it. The lift ranking suggests products that the basket makes more likely, so it is a useful ranking and leads to Phase 3.5 to filter for the lift > 1. 

### Phase 3.5 - Keep only the rules with lift > 1

In [34]:
'''
lift > 1 : the IF products make the THEN product MORE likely  -> worth suggesting
lift = 1 : the IF products change nothing (the THEN product is simply popular)
lift < 1 : the IF products make the THEN product LESS likely
'''
rules_lift = all_rules[all_rules['lift'] > 1]
removed = all_rules[all_rules['lift'] <= 1]
print(f'{len(all_rules):,} rules -> {len(rules_lift):,} kept, {len(removed):,} removed')

# Which products are on the THEN side of the removed rules?
print('\nTHEN side of the removed rules:')
print(readable(removed)['consequents'].value_counts())

28,433 rules -> 26,991 kept, 1,442 removed

THEN side of the removed rules:
consequents
Sac réutilisable    1442
Name: count, dtype: int64


#### Justification of the lift > 1 filter

- The filter removed 1,442 rules and kept 26,991 rules. All the 1,442 removed rules suggest Sac réutilisable on the THEN side, so they are the useless rules found in Phase 3.4. Rules with lift <= 1 are useless because it means that the basket does not make B more likely, because B is bought just as often or less often without A. Recommending it is like suggesting popular items at random.
- Rules with lift > 1 are the right choice because it means A really increases the chance of B. This is the kind of suggestion that can be given to the customers, "Because you have X in your cart, then you may consider adding Y". 
- **Honest Point:** This filter is not perfect, because slightly increasing the filter to > 1.1 can remove further 1,832 bag rules because their lift is just above 1. However, the app will sort the rules by lift, so those rules will rank last. Check the Biscuits screenshot (bag suggested at lift 1.0, "weak"). 

## Phase 4 — Seasonal analysis

### Phase 4.1 - Mine summer (June–August) and winter (December–February) separately

In [35]:
'''
Same pipeline for each season: wide table (build_fast) -> apriori (low_memory) -> rules with lift > 1.
Same thresholds as the year-round rules. Only the simple rules are compared, so the tables stay readable.
'''
month = df['Date'].str[5:7]   # 'YYYY-MM-DD' -> 'MM'
seasons = {'summer': ['06', '07', '08'],
           'winter': ['12', '01', '02']}
season_rules = {}

for name, months in seasons.items():
    season_table = build_fast(df[month.isin(months)])
    season_itemsets = apriori(season_table, min_support=final_support, use_colnames=True, low_memory=True)
    rules = association_rules(season_itemsets, metric='confidence', min_threshold=0.2)
    season_rules[name] = readable(simple_rules(rules[rules['lift'] > 1]))
    print(f'{name}: {len(season_table):,} baskets, {len(season_rules[name])} simple rules with lift > 1')

summer: 252,474 baskets, 335 simple rules with lift > 1
winter: 246,965 baskets, 338 simple rules with lift > 1


### Phase 4.2 - The same rules in both seasons, and the rules that disappear

In [36]:
# Put the summer and winter rules side by side (a rule missing in one season gets NaN there)
compare = season_rules['summer'].merge(season_rules['winter'], on=['antecedents', 'consequents'],
                                       how='outer', suffixes=('_summer', '_winter'))
in_both = compare.dropna()
only_summer = compare[compare['lift_winter'].isna()]
only_winter = compare[compare['lift_summer'].isna()]
print(f'rules in both seasons: {len(in_both)}, only in summer: {len(only_summer)}, only in winter: {len(only_winter)}')

print('\nSame rules in both seasons (strongest in summer first)')
display(in_both.sort_values('lift_summer', ascending=False).head(10))

print('Rules that exist ONLY in summer')
display(only_summer.sort_values('lift_summer', ascending=False).head(10).dropna(axis=1))

print('Rules that exist ONLY in winter')
display(only_winter.sort_values('lift_winter', ascending=False).head(10).dropna(axis=1))

rules in both seasons: 274, only in summer: 61, only in winter: 64

Same rules in both seasons (strongest in summer first)


,antecedents,consequents,support_summer,confidence_summer,lift_summer,support_winter,confidence_winter,lift_winter
85,Charcuteries,Fromage à raclette,0.007,0.329,12.006,0.064,0.819,6.718
165,Fromage à raclette,Charcuteries,0.007,0.266,12.006,0.064,0.525,6.718
164,Fromage à raclette,Bouillon à fondue,0.008,0.275,11.932,0.067,0.552,6.808
55,Bouillon à fondue,Fromage à raclette,0.008,0.326,11.932,0.067,0.830,6.808
106,Cornichons,Fromage à raclette,0.007,0.310,11.340,0.059,0.814,6.679
166,Fromage à raclette,Cornichons,0.007,0.254,11.340,0.059,0.486,6.679
167,Fromage à raclette,Pain baguette rustique,0.006,0.237,10.958,0.055,0.453,6.538
237,Pain baguette rustique,Fromage à raclette,0.006,0.300,10.958,0.055,0.797,6.538
312,Salsa,Croustilles,0.008,0.338,10.558,0.070,0.842,5.016
127,Croustilles,Salsa,0.008,0.239,10.558,0.070,0.415,5.016


Rules that exist ONLY in summer


,antecedents,consequents,support_summer,confidence_summer,lift_summer
319,Sauce BBQ,Relish,0.045,0.328,3.160
38,Bière,Charbon de bois,0.137,0.750,2.780
48,Bière,Steak de boeuf,0.100,0.545,2.744
42,Bière,Pain à hamburger,0.094,0.513,2.724
41,Bière,Ketchup,0.084,0.457,2.711
47,Bière,Saucisses,0.076,0.418,2.693
46,Bière,Sauce BBQ,0.067,0.367,2.667
43,Bière,Relish,0.049,0.270,2.600
240,Pain tranché,Charbon de bois,0.022,0.277,1.027
163,Fromage suisse,Steak de boeuf,0.019,0.204,1.025


Rules that exist ONLY in winter


,antecedents,consequents,support_winter,confidence_winter,lift_winter
53,Bouillon à fondue,Charcuteries,0.040,0.495,6.333
83,Charcuteries,Bouillon à fondue,0.040,0.513,6.333
168,Fromage à raclette,Pommes de terre,0.045,0.368,6.319
54,Bouillon à fondue,Cornichons,0.037,0.458,6.294
104,Cornichons,Bouillon à fondue,0.037,0.510,6.294
105,Cornichons,Charcuteries,0.035,0.486,6.215
84,Charcuteries,Cornichons,0.035,0.452,6.215
56,Bouillon à fondue,Pain baguette rustique,0.034,0.424,6.120
234,Pain baguette rustique,Bouillon à fondue,0.034,0.496,6.120
235,Pain baguette rustique,Charcuteries,0.033,0.476,6.092


In [37]:
# If the website served the year-round rules in January: which year-round rules do not hold in winter?
year_simple = readable(simple_rules(rules_lift))
year_rules = set(zip(year_simple['antecedents'], year_simple['consequents']))
winter_rules = set(zip(season_rules['winter']['antecedents'], season_rules['winter']['consequents']))

print(f'year-round rules that do NOT hold in winter: {len(year_rules - winter_rules)} of {len(year_rules)}')
print(sorted(year_rules - winter_rules)[:10])
print(f'\nwinter rules MISSING from the year-round rules: {len(winter_rules - year_rules)} of {len(winter_rules)}')
print(sorted(winter_rules - year_rules)[:10])

year-round rules that do NOT hold in winter: 30 of 354
[('Algues nori', 'Sac réutilisable'), ('Beurre', 'Sac réutilisable'), ('Bière', 'Charbon de bois'), ('Bière', 'Couches'), ('Bière', 'Ketchup'), ('Bière', 'Pain à hamburger'), ('Bière', 'Sauce BBQ'), ('Bière', 'Saucisses'), ('Bière', 'Steak de boeuf'), ('Café moulu', 'Sac réutilisable')]

winter rules MISSING from the year-round rules: 14 of 338
[('Bananes', 'Sac réutilisable'), ('Bière', 'Bretzels'), ('Bière', 'Croustilles de maïs'), ('Bière', 'Salsa'), ('Charcuteries', 'Sac réutilisable'), ('Croustilles', 'Sac réutilisable'), ('Crème glacée', 'Sac réutilisable'), ('Lait 2%', 'Sac réutilisable'), ('Lingettes pour bébé', 'Sac réutilisable'), ('Maïs soufflé', 'Sac réutilisable')]


### Phase 4.3 - Conclusion

- Summer and winter situations are different. 274 rules hold in both seasons, 61 in summer only and 64 in winter only. So about 1 in 5 rules are seasonal only. In summer, beer → BBQ items occupy 7 of the top 10 summer-only rules by lift. In winter, Bouillon à fondue → Charcuteries and vice versa are the top winter-only rules. Charcuteries, Bouillon à fondue, Cornichons and Pain baguette rustique are items bought together in winter that make each other be bought more often. 
- Shared rules in both seasons may have different strength in each season. For example, Charcuteries → Fromage à raclette has a support of 0.007 in summer vs 0.064 in winter (about 9× more baskets), and confidence 0.33 vs 0.82. Its lift is actually higher in summer (12.0 vs 6.7). Fromage à raclette are rare in summer, so the few buyers buy them together. This shows lift alone can be misleading when support is tiny.
- Year-round rules may not hold in winter. Notably, Bière → BBQ items (Charbon de bois, Steak, Ketchup, Saucisses, Sauce BBQ…) fail in winter. The reason that it shows up in January is because the recommender looks at the basket content, not the date. The summer baskets with Bière weigh enough to push the rule through the thresholds for the whole year. If the website served the year-round rules in January, a beer buyer would be told that customers buy Charbon de bois "45% of the time", but in January only 15% of beer buyers do, so the explanation would be about 3× wrong. The year-round rules would also miss the winter beer rules (Bière → Bretzels, Salsa, Croustilles de maïs).
- How often should the company re-mine? At least once per season or monthly with a rolling window of recent months if possible. This is because the rules change between seasons. If the data is available, re-mining costs almost nothing: it takes only about 26 s with low_memory = True. 

## Phase 5 — Save the rules (offline mining, online app)

In [38]:
'''
The app never calls Apriori: it only loads these two files.
- rules.json    : the rules with lift > 1 and ONE product on the THEN side (the app suggests one product at a time)
- metadata.json : thresholds, mining date, number of rules, coverage, and the support of every product
                  (the app uses it for the "most popular products" fallback and to explain rare products)
'''
import json
from datetime import date

final_rules = rules_lift[rules_lift['consequents'].apply(len) == 1].sort_values('lift', ascending=False)

# frozensets cannot be saved to JSON, so each rule becomes a plain dictionary
rules_list = []
for _, rule in final_rules.iterrows():
    rules_list.append({'antecedents': sorted(rule['antecedents']),
                       'consequent': list(rule['consequents'])[0],
                       'support': round(rule['support'], 6),
                       'confidence': round(rule['confidence'], 6),
                       'lift': round(rule['lift'], 4)})

# Coverage: the products that are on the THEN side of at least one rule can be recommended
recommendable = sorted({rule['consequent'] for rule in rules_list})
never_recommended = sorted(set(products['Item']) - set(recommendable))

metadata = {
    'mining_date': date.today().isoformat(),
    'data': {'baskets': len(basket_table), 'products': len(products)},
    'thresholds': {'min_support': final_support, 'min_confidence': 0.2, 'min_lift': 1.0, 'low_memory': True},
    'n_rules': len(rules_list),
    'coverage': {'recommendable_products': len(recommendable),
                 'total_products': len(products),
                 'never_recommended': never_recommended},
    'item_support': item_freq.round(6).to_dict(),   # support of every product, from Phase 1.3
}

with open('../models/rules.json', 'w', encoding='utf-8') as file:
    json.dump(rules_list, file, ensure_ascii=False, indent=1)
with open('../models/metadata.json', 'w', encoding='utf-8') as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print(f'saved {len(rules_list):,} rules to models/rules.json')
print(f'coverage: {len(recommendable)} of {len(products)} products can ever be recommended')
print(f'never recommended ({len(never_recommended)}): {never_recommended}')

saved 11,059 rules to models/rules.json
coverage: 58 of 98 products can ever be recommended
never recommended (40): ['Bagels', 'Baguette', 'Beurre', "Beurre d'arachide", 'Biscuits', 'Carottes', 'Caviar', 'Chocolat noir', 'Cidre de glace', 'Confiture de fraises', 'Crevettes', 'Crème glacée', 'Céréales', 'Détergent à lessive', 'Eau pétillante', 'Foie gras', 'Fromage cheddar', 'Fromage suisse', 'Huile de truffe', 'Lait 2%', 'Moutarde de Dijon', 'Noix mélangées', 'Oignons', 'Pain tranché', 'Papier essuie-tout', 'Papier hygiénique', 'Petits pois surgelés', 'Pizza surgelée', 'Poitrine de poulet', 'Pommes', 'Riz basmati', 'Sacs à ordures', 'Safran', 'Savon à vaisselle', 'Sucre blanc', 'Thon en conserve', 'Thé vert', 'Tortillas', 'Vinaigre balsamique vieilli', 'Yogourt grec']


## Recommendation engine

The engine is in `app/streamlit_app.py`, in the `recommend()` function. It is adapted from the recommender built in the 09-22 Association Rules exercise: subset matching on the IF side, products already in the basket skipped, the highest-lift rule kept per product, and suggestions sorted by lift. It reads `models/rules.json` and never calls Apriori.